In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip install pandas faker numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 28.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

# Initialize Faker with a seed for reproducible results
fake = Faker()
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# Configuration Parameters
NUM_CUSTOMERS = 300
NUM_PRODUCTS = 35
NUM_STORES = 10
NUM_SALES_RECORDS = 300000
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 12, 31)

# ---------------------------------------------------------
# 1. GENERATE MASTER DATA
# ---------------------------------------------------------

# Master: Customers
customers = []
for i in range(1, NUM_CUSTOMERS + 1):
    customers.append({
        "customer_id": f"CUST-{i:04d}",
        "customer_name": fake.name(),
        "email": fake.email(),
        "signup_date": fake.date_between(start_date="-2y", end_date="today"),
        "city": fake.city(),
        "state": fake.state_abbr()
    })
df_customers = pd.DataFrame(customers)

# Master: Products
categories = {
    "Electronics": (50, 500),
    "Clothing": (15, 80),
    "Home & Kitchen": (10, 150),
    "Books": (8, 30)
}

products = []
for i in range(1, NUM_PRODUCTS + 1):
    category = random.choice(list(categories.keys()))
    min_price, max_price = categories[category]
    cost = round(random.uniform(min_price, max_price), 2)
    # Unit price set at a 20%-50% markup over cost
    unit_price = round(cost * random.uniform(1.2, 1.5), 2)

    products.append({
        "product_id": f"PROD-{i:03d}",
        "product_name": f"{fake.word().capitalize()} {category[:-1]}",
        "category": category,
        "cost": cost,
        "unit_price": unit_price
    })
df_products = pd.DataFrame(products)

# Master: Stores
stores = []
for i in range(1, NUM_STORES + 1):
    stores.append({
        "store_id": f"STORE-{i:02d}",
        "store_name": f"{fake.city()} Branch",
        "region": random.choice(["North", "South", "East", "West"]),
        "store_type": random.choice(["Retail", "Outlet", "Flagship"])
    })
df_stores = pd.DataFrame(stores)

# ---------------------------------------------------------
# 2. GENERATE SALES TRANSACTION DATA
# ---------------------------------------------------------

def random_date(start, end):
    delta = end - start
    random_days = random.randint(0, delta.days)
    # Output formatted date string directly (YYYY-MM-DD)
    return (start + timedelta(days=random_days)).strftime("%Y-%m-%d")

sales = []
for i in range(1, NUM_SALES_RECORDS + 1):
    # Select foreign keys from master data
    cust = df_customers.sample(1).iloc[0]
    prod = df_products.sample(1).iloc[0]
    store = df_stores.sample(1).iloc[0]

    quantity = random.choices([1, 2, 3, 4, 5], weights=[50, 25, 15, 7, 3])[0]
    unit_price = prod["unit_price"]
    discount = round(random.choice([0.0, 0.0, 0.0, 0.05, 0.10, 0.15]), 2)

    # Calculate derived financial figures
    gross_amount = round(quantity * unit_price, 2)
    discount_amount = round(gross_amount * discount, 2)
    net_amount = round(gross_amount - discount_amount, 2)

    sales.append({
        "transaction_id": f"TXN-{i:06d}",
        "transaction_date": random_date(START_DATE, END_DATE),
        "customer_id": cust["customer_id"],
        "product_id": prod["product_id"],
        "store_id": store["store_id"],
        "quantity": quantity,
        "unit_price": unit_price,
        "gross_amount": gross_amount,
        "discount_pct": discount,
        "discount_amount": discount_amount,
        "net_amount": net_amount,
        "payment_method": random.choice(["Credit Card", "Debit Card", "Cash", "Digital Wallet"])
    })

df_sales = pd.DataFrame(sales)

# Sort transactions by date for realistic ordering
df_sales = df_sales.sort_values(by="transaction_date").reset_index(drop=True)

# ---------------------------------------------------------
# 3. EXPORT TO CSV / VIEW DATA
# ---------------------------------------------------------

# Save to CSV files
df_customers.to_csv("/content/drive/MyDrive/data/master_customers.csv", index=False)
df_products.to_csv("/content/drive/MyDrive/data/master_products.csv", index=False)
df_stores.to_csv("/content/drive/MyDrive/data/master_stores.csv", index=False)
df_sales.to_csv("/content/drive/MyDrive/data/fact_sales.csv", index=False)

print("Data generation complete! Saved 4 datasets:")
print(f" - Customers: {len(df_customers)} records")
print(f" - Products:  {len(df_products)} records")
print(f" - Stores:    {len(df_stores)} records")
print(f" - Sales:     {len(df_sales)} records")

Data generation complete! Saved 4 datasets:
 - Customers: 300 records
 - Products:  35 records
 - Stores:    10 records
 - Sales:     300000 records
